# Industrial Modbus Threat Hunting

Use **Zeek + CISA ICSNPP-Modbus** output from prerecorded PCAPs to explore industrial control values over time.


## 1. Select a dataset and load the Zeek evidence

Set the dataset name below. For example, use `capture1` or `capture2` for the exercise. The corresponding `output/<dataset>/modbus_detailed.log` must already exist.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

# -------- EDIT HERE --------
dataset = "capture1"  # Update to change to other datasets after Zeek processing
# ---------------------------

here = Path.cwd()
root = here.parent if here.name == "notebooks" else here
log_file = root / "output" / dataset / "modbus_detailed.log"
records = pd.read_json(log_file, lines=True, precise_float=True)
expected = {"ts", "func", "address", "request_values", "response_values"}
records["time_utc"] = pd.to_datetime(records["ts"], unit="s", utc=True)
records = records.sort_values("ts").reset_index(drop=True)
print(f"Dataset: {dataset}  |  Modbus detailed records: {len(records):,}")
print(f"Period: {records['time_utc'].min()} to {records['time_utc'].max()} (UTC)")

## 2 · Discover the industrial protocol operations

Before examining any particular signal, identify **what data the capture contains**. Each function provides different evidence:

| Modbus function | Direction of industrial information | Values recorded by this Zeek extension |
|---|---|---|
| `READ_DISCRETE_INPUTS` (FC2) | PLC reads binary input states | `response_values` (0/1) |
| `READ_INPUT_REGISTERS` (FC4) | PLC reads numerical input values | `response_values` |
| `WRITE_MULTIPLE_COILS` (FC15) | PLC sends binary output commands | `request_values` (0/1) |
| `WRITE_MULTIPLE_REGISTERS` (FC16) | PLC sends numerical output commands | `request_values` |

In [ ]:
inventory = (
    records.groupby("func", dropna=False)
    .agg(records=("ts", "size"), first_seen=("time_utc", "min"), last_seen=("time_utc", "max"))
    .reset_index()
    .sort_values("func")
)
display(inventory)

def value_field(function):
    if str(function).startswith("READ_"):
        return "response_values"
    if str(function).startswith("WRITE_"):
        return "request_values"
    return None

def is_sequence(x):
    return isinstance(x, (list, tuple, np.ndarray)) and len(x) > 0

summary = []
for function, group in records.groupby("func", dropna=False):
    field = value_field(function)
    examples = group.loc[group[field].map(is_sequence)] if field in group else group.iloc[:0]
    first = examples.iloc[0] if not examples.empty else None
    summary.append({
        "function": function,
        "total_records": len(group),
        "records_with_values": len(examples),
        "value_field": field,
        "starting_address_example": None if first is None else first.get("address"),
        "values_example": None if first is None else first[field],
    })
print("Where can we find usable signal values?")
display(pd.DataFrame(summary))

## 3 · Choose a Modbus operation and signal addresses

**Change the parameters below to examine *different* inputs and outputs.**

- Start by exploring **discrete inputs** (FC2) or **coil commands** (FC15). These should contain binary states, often 0 or 1.
- Next explore **input registers** (FC4) and **holding-register writes** (FC16), which contain numeric values.
- `addresses = None` means **show all addresses observed for that operation**. Alternatively set, for example, `addresses = [0, 1]` or `addresses = [2]`.
- Set `signed16 = True` *only* where the PLC tag represents a signed 16-bit value.
- Set `scale_factor = 1` *only* where a documented tag mapping says 100 raw units represent 1 engineering unit. Otherwise use `1`.


In [ ]:
# ==================== EDIT HERE ====================
function = "READ_DISCRETE_INPUTS"  # Try READ_INPUT_REGISTERS, WRITE_MULTIPLE_COILS, WRITE_MULTIPLE_REGISTERS
addresses = None                 # None = all observed addresses; e.g. [0], [0, 1], or [2]
signed16 = False                 # True for KNOWN signed 16-bit register tags; not for coils
scale_factor = 100                 # 1 = raw values; 100 only with verified engineering mapping
# ===================================================

value_column = value_field(function)
print(f"Investigating: {function} | field: {value_column} | selected addresses: {addresses}")

## 4 · Extract the selected signals into a structured table

This step is used for graphing and statistical analysis.

Read the code: `base_address + offset` identifies each signal. The raw data remain unchanged; optional signed decoding or scaling is an additional interpretation.

In [ ]:
def signed_16(value):
    value = int(value)
    return value - 65536 if value >= 32768 else value

def extract_signals(source, selected_function, selected_addresses=None, decode_signed=False, factor=1):
    field = value_field(selected_function)
    out = []
    for _, event in source.loc[source["func"].eq(selected_function)].iterrows():
        values = event.get(field)
        if not is_sequence(values):
            continue
        base = event.get("address")
        if pd.isna(base):
            continue
        for offset, raw in enumerate(values):
            addr = int(base) + offset
            if selected_addresses is not None and addr not in selected_addresses:
                continue
            if raw is None or pd.isna(raw):
                continue
            raw_int = int(raw)
            decoded = signed_16(raw_int) if decode_signed else raw_int
            out.append({
                "ts": float(event["ts"]),
                "time_utc": event["time_utc"],
                "function": selected_function,
                "address": addr,
                "raw": raw_int,
                "value": decoded / factor,
                "tid": event.get("tid"),
                "uid": event.get("uid"),
            })
    return pd.DataFrame(out)

signals = extract_signals(records, function, addresses, signed16, scale_factor)
signals = signals.sort_values(["ts", "address"]).reset_index(drop=True)
print(f"Extracted {len(signals):,} signal observations from {signals['ts'].nunique():,} time points.")
print(f"Available addresses in your selection: {sorted(signals['address'].unique().tolist())}")
display(signals[["time_utc", "function", "address", "raw", "value", "tid"]].head(12))

## 5 · Inspect signal behaviour before plotting

For each address, look at its observed value range, number of distinct values and transitions. 

In [ ]:
stats = (
    signals.groupby("address")["value"]
    .agg(observations="size", distinct_values="nunique", minimum="min", maximum="max", first="first", last="last")
    .reset_index()
)
display(stats.round(3))

## 6 · Visualise selected addresses over time

- Binary signals: inspect ON/OFF transitions, including their order and duration.
- Numerical signals: inspect value magnitude and timing; apply scaling only when justified.

**Try changing `function` and `addresses` in Section 3**, then rerun the subsequent cells.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for addr, part in signals.groupby("address"):
    part = part.sort_values("ts")
    ax.plot(part["time_utc"], part["value"], label=f"Address {addr}", linewidth=1.5,
            drawstyle="steps-post" if "COILS" in function or "DISCRETE_INPUTS" in function else "default")
ax.set(title=f"{function} — {dataset}", xlabel="Packet timestamp (UTC)",
       ylabel=f"Value ({'signed' if signed16 else 'raw'} / {scale_factor})")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S", tz=mdates.UTC))
if "COILS" in function or "DISCRETE_INPUTS" in function:
    ax.set_yticks([0, 1])
    ax.set_ylim(-0.12, 1.12)
ax.grid(alpha=0.25)
ax.legend(title="Modbus signal", loc="best", ncol=2 if signals['address'].nunique()>4 else 1)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7 · Identify value transitions in packets

An industrial controller can send the same values many times per second. Filter repeated observations to identify **when the values actually change**.

In [ ]:
signals["prior_raw"] = signals.groupby("address")["raw"].shift(1)
changes = signals.loc[signals["prior_raw"].isna() | signals["raw"].ne(signals["prior_raw"])].copy()
change_counts = changes.groupby("address").size().rename("initial_plus_changes").reset_index()
print(f"{len(changes):,} first observations or change points, across {signals['address'].nunique()} address(es).")
display(change_counts)

# Try different value of the head(value)
display(changes[["time_utc", "address", "prior_raw", "raw", "value", "tid"]].head(40))
print("Showing first 40 rows. For additional rows change .head(40) or use the filter in the next exercise.")